In [3]:
from agents import Agent, Runner
from dotenv import load_dotenv
from pydantic import BaseModel
import gradio as gr


In [4]:
load_dotenv()

True

In [9]:
from pydantic import BaseModel

class JobAnalysis(BaseModel):
    resume_skills: list[str]
    required_skills: list[str]
    matching_skills: list[str]
    missing_skills: list[str]

In [3]:
#Create first agent


agent = Agent(
    name ="Job-analysis agent",
    instructions = 
    """
    Analyzes the student’s resume and target job description,

    Extracts skills, education, projects, and work experience,

    Identifies job requirements and qualifications,

    Finds matches and potential skill gaps""",

    model ="gpt-4o-mini"
)

In [4]:

resume = """ 
I am a dot net developer with 5+ years expeirence,
I am profficient in C# and react for front end, also SQU and Mongo db database

"""
job_description = """
 we are looking for a dot net developer with 4+ years of expirience also with anglular expert
"""
input_string =  f""" resume:
{resume} 

job description:
 {job_description}.
 """

In [5]:
result1 = await Runner.run(agent, input_string)
print(result1.final_output)

Let's analyze the provided resume and job description:

### **Resume Analysis:**

**Skills:**
- Proficient in C#
- Proficient in React for front-end development
- Experience with SQL databases
- Experience with MongoDB

**Experience:**
- 5+ years as a .NET developer

### **Job Description Requirements:**

**Qualifications:**
- .NET developer with 4+ years of experience
- Expertise in Angular
  
### **Matches:**
- The candidate meets the experience requirement (5+ years).
- The candidate has relevant skills in .NET (C#).

### **Skill Gaps:**
- The job requires expertise in Angular, which the candidate does not mention.
- While the candidate has skills in React, it does not align with the requirement for Angular.

### **Summary:**
- The candidate largely meets the experience and a primary skill requirement but lacks specific knowledge in Angular as required by the job description. It would be beneficial for the candidate to either gain experience in Angular or highlight any transferable 

In [6]:
agent =Agent[any](
    name="Interview-agent",
    instructions="""
    create interview questions,
    ask personalized questions to the job seaker
    
    """,
    model ="gpt-4o-mini"
)

In [7]:
result = await Runner.run(agent, "ASP.NET interview questions")
print(result.final_output)

Here are some ASP.NET interview questions that you can use, along with a few personalized questions to engage with the job seeker:

### Technical ASP.NET Questions

1. **What is the difference between ASP.NET Web Forms and ASP.NET MVC?**
2. **Can you explain the ASP.NET page lifecycle?**
3. **What is the purpose of Global.asax in an ASP.NET application?**
4. **How do you handle errors in ASP.NET?**
5. **What are ViewState and how does it work?**
6. **What are the differences between Server.Transfer and Response.Redirect?**
7. **Explain the concept of model binding in ASP.NET MVC.**
8. **What is Dependency Injection, and how would you implement it in ASP.NET Core?**
9. **How does the ASP.NET pipeline process requests?**
10. **Can you explain authentication and authorization in ASP.NET?**

### Personalized Questions

1. **Can you tell me about a project where you used ASP.NET? What were the challenges you faced?**
2. **What specific features or tools in ASP.NET do you find most beneficia

In [ ]:
import gradio as gr
from openai import OpenAI
from pathlib import Path
import uuid

client = OpenAI()


# ----------------------------------------
# Convert interview question to voice
# ----------------------------------------

def text_to_speech(text):

    audio_file = f"question_{uuid.uuid4()}.mp3"

    with client.audio.speech.with_streaming_response.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=text
    ) as response:

        response.stream_to_file(audio_file)

    return audio_file


# ----------------------------------------
# Start Interview
# ----------------------------------------

def start_interview(resume, job_description):

    if resume is None:
        return "Please upload your resume.", None

    if job_description is None:
        return "Please upload the job description.", None

    # Later this question will come from your Interviewer Agent
    question = (
        "Welcome to your interview. "
        "Let's begin. Tell me about yourself and "
        "how your experience relates to this position."
    )

    # Convert question to voice
    question_audio = text_to_speech(question)

    return question, question_audio


# ----------------------------------------
# Receive candidate voice answer
# ----------------------------------------

def submit_answer(audio):

    if audio is None:
        return "Please record your answer first."

    return "Your answer was received."


# ========================================
# GRADIO UI
# ========================================

with gr.Blocks(title="AI Interview Coach") as demo:

    gr.Markdown(
        """
        # 🎤 AI Interview Coach

        Upload your **Resume** and **Job Description**,
        then start your personalized voice interview.
        """
    )

    # ------------------------------------
    # Upload Documents
    # ------------------------------------

    with gr.Row():

        resume_file = gr.File(
            label="📄 Upload Resume",
            file_types=[".pdf", ".docx", ".txt"]
        )

        jd_file = gr.File(
            label="📋 Upload Job Description",
            file_types=[".pdf", ".docx", ".txt"]
        )


    start_button = gr.Button(
        "▶ Start Interview",
        variant="primary"
    )


    # ------------------------------------
    # AI Interviewer
    # ------------------------------------

    gr.Markdown("## 🤖 AI Interviewer")

    question_output = gr.Textbox(
        label="Interview Question",
        lines=3,
        interactive=False
    )

    # AI speaks the question
    question_voice = gr.Audio(
        label="🔊 Interviewer Voice",
        autoplay=True,
        interactive=False
    )


    # ------------------------------------
    # Candidate Answer
    # ------------------------------------

    gr.Markdown("## 🎙️ Your Answer")

    voice_input = gr.Audio(
        sources=["microphone"],
        type="filepath",
        label="Record Your Answer"
    )


    submit_button = gr.Button(
        "Submit Answer",
        variant="primary"
    )


    # ------------------------------------
    # Feedback
    # ------------------------------------

    feedback_output = gr.Textbox(
        label="AI Coach Feedback",
        lines=5,
        interactive=False
    )


    # ------------------------------------
    # EVENTS
    # ------------------------------------

    start_button.click(
        fn=start_interview,
        inputs=[
            resume_file,
            jd_file
        ],
        outputs=[
            question_output,
            question_voice
        ]
    )


    submit_button.click(
        fn=submit_answer,
        inputs=voice_input,
        outputs=feedback_output
    )


demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
